# Global raster alignment audit (read-only)

This notebook checks **structural and identity alignment** for MODIS and Sentinel-1 without loading full rasters into RAM. It does not change any files.

Important distinction: matching row counts, unique IDs, and valid row indices do not by themselves prove that a MODIS sidecar row points to the exact patch written at that Zarr index. The source-code scan in Cell 3 looks for the original writer so we can verify how the sidecar and arrays were written together.

In [1]:
from pathlib import Path
import json
import os
import re
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import duckdb
import zarr

ROOT = Path.home() / "Code/UG_SM/Data/Global"
YEARS = list(range(2015, 2027))
CANDIDATES = ROOT / "station_date_candidates.parquet"
MODIS_ROOT = ROOT / "modis/evi_lst_patch_dataset"
S1_ROOT = ROOT / "sentinel1/nearest_valid_overview_v1"

if not ROOT.exists():
    raise FileNotFoundError(f"Data root not found: {ROOT}. Edit ROOT at the top of this cell.")
if not CANDIDATES.exists():
    raise FileNotFoundError(f"Canonical sample file not found: {CANDIDATES}")

con_audit = duckdb.connect(":memory:")
con_audit.execute("SET memory_limit='512MB'")
con_audit.execute("SET threads=2")

def sqlq(value):
    return "'" + str(value).replace("'", "''") + "'"

def zarr_arrays_recursive(group, prefix=""):
    """Yield (name, array) pairs for both Zarr v2/v3 group APIs."""
    try:
        array_names = list(group.array_keys())
    except Exception:
        array_names = []
    for name in sorted(array_names):
        arr = group[name]
        yield (f"{prefix}{name}", arr)
    try:
        group_names = list(group.group_keys())
    except Exception:
        group_names = []
    for name in sorted(group_names):
        yield from zarr_arrays_recursive(group[name], prefix=f"{prefix}{name}/")

def inspect_store(store_path, expected_rows, label):
    """Read only a few sample rows per array; never materializes a whole store."""
    if not store_path.exists():
        return {"store": label, "zarr_exists": False, "expected_rows": expected_rows,
                "arrays": 0, "arrays_axis0_match": False, "sample_reads_ok": 0,
                "sample_reads_tested": 0, "error": f"Missing store: {store_path}"}
    try:
        group = zarr.open_group(str(store_path), mode="r")
        arrays = list(zarr_arrays_recursive(group))
        if not arrays:
            raise ValueError("No arrays found in Zarr store")
        axis_matches = all(len(a.shape) > 0 and int(a.shape[0]) == int(expected_rows)
                           for _, a in arrays)
        test_rows = sorted(set([0, int(expected_rows)//2, int(expected_rows)-1])) if expected_rows else []
        tested = ok = 0
        errors = []
        for name, arr in arrays:
            for idx in test_rows:
                tested += 1
                try:
                    value = np.asarray(arr[idx])  # slice just one sample, not full array
                    if value.size == 0:
                        raise ValueError("empty sample slice")
                    ok += 1
                except Exception as exc:
                    errors.append(f"{name}[{idx}]: {type(exc).__name__}: {exc}")
        return {
            "store": label, "zarr_exists": True, "expected_rows": int(expected_rows),
            "arrays": len(arrays), "arrays_axis0_match": bool(axis_matches),
            "sample_reads_ok": ok, "sample_reads_tested": tested,
            "array_shapes": "; ".join(f"{name}={tuple(arr.shape)}" for name, arr in arrays),
            "error": " | ".join(errors[:3]) if errors else None,
        }
    except Exception as exc:
        return {"store": label, "zarr_exists": True, "expected_rows": expected_rows,
                "arrays": 0, "arrays_axis0_match": False, "sample_reads_ok": 0,
                "sample_reads_tested": 0,
                "error": f"{type(exc).__name__}: {exc}"}

candidate_q = sqlq(CANDIDATES.resolve())
modis_rows = []
s1_rows = []
array_read_rows = []

for year in YEARS:
    modis_sidecar = MODIS_ROOT / str(year) / "samples.parquet"
    modis_store = MODIS_ROOT / str(year) / "patches.zarr"
    s1_index = S1_ROOT / f"samples_nearest_valid_{year}.parquet"
    s1_store = S1_ROOT / f"s1_nearest_valid_{year}.zarr"

    # ---- MODIS: verify the sidecar's IDs and identity against canonical samples.
    if not modis_sidecar.exists():
        modis_rows.append({"year": year, "status": "FAIL", "reason": "sidecar missing"})
    else:
        sp = sqlq(modis_sidecar.resolve())
        n, n_ids, null_ids = con_audit.execute(f"""
            SELECT COUNT(*), COUNT(DISTINCT sample_id),
                   COUNT(*) FILTER (WHERE sample_id IS NULL)
            FROM read_parquet({sp})
        """).fetchone()
        zarr_row_count = int(n)
        set_mismatch = con_audit.execute(f"""
            SELECT COUNT(*) FROM (
                SELECT s.sample_id
                FROM read_parquet({sp}) s
                LEFT JOIN read_parquet({candidate_q}) c USING (sample_id)
                WHERE c.sample_id IS NULL
                   OR EXTRACT(YEAR FROM CAST(c.sample_date AS DATE)) <> {year}
                UNION ALL
                SELECT c.sample_id
                FROM read_parquet({candidate_q}) c
                LEFT JOIN read_parquet({sp}) s USING (sample_id)
                WHERE EXTRACT(YEAR FROM CAST(c.sample_date AS DATE)) = {year}
                  AND s.sample_id IS NULL
            ) x
        """).fetchone()[0]
        identity_mismatches = con_audit.execute(f"""
            SELECT COUNT(*)
            FROM read_parquet({sp}) s
            JOIN read_parquet({candidate_q}) c USING (sample_id)
            WHERE TRIM(CAST(s.station_id AS VARCHAR)) IS DISTINCT FROM
                  TRIM(CAST(c.station_id AS VARCHAR))
               OR CAST(s.sample_date AS DATE) IS DISTINCT FROM CAST(c.sample_date AS DATE)
        """).fetchone()[0]
        zarr_ok = False
        zarr_error = None
        if modis_store.exists():
            result = inspect_store(modis_store, zarr_row_count, f"MODIS {year}")
            array_read_rows.append(result)
            zarr_ok = bool(result.get("zarr_exists") and result.get("arrays_axis0_match")
                           and result.get("sample_reads_ok") == result.get("sample_reads_tested"))
            zarr_error = result.get("error")
        modis_pass = (n == n_ids == zarr_row_count and null_ids == 0 and
                      set_mismatch == 0 and identity_mismatches == 0 and zarr_ok)
        modis_rows.append({
            "year": year, "sidecar_rows": int(n), "unique_ids": int(n_ids),
            "null_ids": int(null_ids), "canonical_set_mismatches": int(set_mismatch),
            "station_date_mismatches": int(identity_mismatches),
            "zarr_axis0_and_sample_reads_ok": zarr_ok,
            "status": "PASS_STRUCTURAL_ONLY" if modis_pass else "FAIL",
            "note": "Row-order-to-patch identity is NOT proven by these checks."
                   + (f" Zarr note: {zarr_error}" if zarr_error else "")
        })

    # ---- Sentinel-1: validate explicit sample_id -> row_index mappings.
    if not s1_index.exists():
        s1_rows.append({"year": year, "status": "FAIL", "reason": "index missing"})
    else:
        ip = sqlq(s1_index.resolve())
        cols = set(pq.ParquetFile(s1_index).schema_arrow.names)
        required = {"sample_id", "station_id", "sample_date", "year", "row_index"}
        if not required.issubset(cols):
            s1_rows.append({"year": year, "status": "FAIL",
                            "reason": f"missing columns {sorted(required-cols)}"})
        else:
            n, n_ids, n_indices, lo, hi, null_indices, bad_years = con_audit.execute(f"""
                SELECT COUNT(*), COUNT(DISTINCT sample_id), COUNT(DISTINCT row_index),
                       MIN(row_index), MAX(row_index),
                       COUNT(*) FILTER (WHERE row_index IS NULL),
                       COUNT(*) FILTER (WHERE CAST(year AS INTEGER) <> {year}
                          OR EXTRACT(YEAR FROM CAST(sample_date AS DATE)) <> {year})
                FROM read_parquet({ip})
            """).fetchone()
            set_mismatch = con_audit.execute(f"""
                SELECT COUNT(*) FROM (
                    SELECT s.sample_id FROM read_parquet({ip}) s
                    LEFT JOIN read_parquet({candidate_q}) c USING (sample_id)
                    WHERE c.sample_id IS NULL
                       OR EXTRACT(YEAR FROM CAST(c.sample_date AS DATE)) <> {year}
                    UNION ALL
                    SELECT c.sample_id FROM read_parquet({candidate_q}) c
                    LEFT JOIN read_parquet({ip}) s USING (sample_id)
                    WHERE EXTRACT(YEAR FROM CAST(c.sample_date AS DATE)) = {year}
                      AND s.sample_id IS NULL
                ) x
            """).fetchone()[0]
            identity_mismatches = con_audit.execute(f"""
                SELECT COUNT(*)
                FROM read_parquet({ip}) s
                JOIN read_parquet({candidate_q}) c USING (sample_id)
                WHERE TRIM(CAST(s.station_id AS VARCHAR)) IS DISTINCT FROM
                      TRIM(CAST(c.station_id AS VARCHAR))
                   OR CAST(s.sample_date AS DATE) IS DISTINCT FROM CAST(c.sample_date AS DATE)
            """).fetchone()[0]
            zarr_ok = False
            zarr_error = None
            if s1_store.exists():
                result = inspect_store(s1_store, int(n), f"Sentinel-1 {year}")
                array_read_rows.append(result)
                zarr_ok = bool(result.get("zarr_exists") and result.get("arrays_axis0_match")
                               and result.get("sample_reads_ok") == result.get("sample_reads_tested"))
                zarr_error = result.get("error")
            contiguous = bool(n and lo == 0 and hi == n - 1 and n_indices == n)
            index_pass = (n == n_ids == n_indices and null_indices == 0 and
                          bad_years == 0 and set_mismatch == 0 and
                          identity_mismatches == 0 and contiguous and zarr_ok)
            s1_rows.append({
                "year": year, "index_rows": int(n), "unique_ids": int(n_ids),
                "unique_row_indices": int(n_indices), "min_row_index": lo,
                "max_row_index": hi, "null_row_indices": int(null_indices),
                "bad_years": int(bad_years), "canonical_set_mismatches": int(set_mismatch),
                "station_date_mismatches": int(identity_mismatches),
                "contiguous_zero_based_index": contiguous,
                "zarr_axis0_and_sample_reads_ok": zarr_ok,
                "status": "PASS_INDEX_CHECKS" if index_pass else "FAIL",
                "note": "Index metadata checks pass; semantic alignment still depends on the index being written with the Zarr rows."
                       + (f" Zarr note: {zarr_error}" if zarr_error else "")
            })

modis_df = pd.DataFrame(modis_rows)
s1_df = pd.DataFrame(s1_rows)
array_df = pd.DataFrame(array_read_rows)

print("MODIS structural / identity checks")
display(modis_df)
print("Sentinel-1 index / identity checks")
display(s1_df)
print("Zarr metadata and sampled-row read checks")
display(array_df)

# Keep this as an audit, not a hard stop: source-level evidence is needed for MODIS semantics.
print(); print("Interpretation:")
print("- Any FAIL means fix the corresponding mismatch before training with that raster.")
print("- PASS_STRUCTURAL_ONLY for MODIS is NOT proof that row i in samples.parquet is the patch at Zarr row i.")
print("- A valid Sentinel-1 row_index is only trustworthy if its producer records the same row index while writing the Zarr arrays.")


/home/mayur/Code/UG_SM/.venv/lib/python3.14/site-packages/zarr/core/group.py:3413: ZarrUserWarning: Object at zarr.json:Zone.Identifier is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


MODIS structural / identity checks


,year,sidecar_rows,unique_ids,null_ids,canonical_set_mismatches,station_date_mismatches,zarr_axis0_and_sample_reads_ok,status,note
0,2015,329903,329903,0,0,0,True,PASS_STRUCTURAL_ONLY,Row-order-to-patch identity is NOT proven by t...
1,2016,394790,394790,0,0,0,True,PASS_STRUCTURAL_ONLY,Row-order-to-patch identity is NOT proven by t...
2,2017,369953,369953,0,0,0,True,PASS_STRUCTURAL_ONLY,Row-order-to-patch identity is NOT proven by t...
3,2018,344806,344806,0,0,0,True,PASS_STRUCTURAL_ONLY,Row-order-to-patch identity is NOT proven by t...
4,2019,308679,308679,0,0,0,True,PASS_STRUCTURAL_ONLY,Row-order-to-patch identity is NOT proven by t...
5,2020,269853,269853,0,0,0,True,PASS_STRUCTURAL_ONLY,Row-order-to-patch identity is NOT proven by t...
6,2021,259625,259625,0,0,0,True,PASS_STRUCTURAL_ONLY,Row-order-to-patch identity is NOT proven by t...
7,2022,260490,260490,0,0,0,True,PASS_STRUCTURAL_ONLY,Row-order-to-patch identity is NOT proven by t...
8,2023,233307,233307,0,0,0,True,PASS_STRUCTURAL_ONLY,Row-order-to-patch identity is NOT proven by t...
9,2024,264946,264946,0,0,0,True,PASS_STRUCTURAL_ONLY,Row-order-to-patch identity is NOT proven by t...


Sentinel-1 index / identity checks


,year,index_rows,unique_ids,unique_row_indices,min_row_index,max_row_index,null_row_indices,bad_years,canonical_set_mismatches,station_date_mismatches,contiguous_zero_based_index,zarr_axis0_and_sample_reads_ok,status,note
0,2015,329903,329903,329903,0,329902,0,0,0,0,True,True,PASS_INDEX_CHECKS,Index metadata checks pass; semantic alignment...
1,2016,394790,394790,394790,0,394789,0,0,0,0,True,True,PASS_INDEX_CHECKS,Index metadata checks pass; semantic alignment...
2,2017,369953,369953,369953,0,369952,0,0,0,0,True,True,PASS_INDEX_CHECKS,Index metadata checks pass; semantic alignment...
3,2018,344806,344806,344806,0,344805,0,0,0,0,True,True,PASS_INDEX_CHECKS,Index metadata checks pass; semantic alignment...
4,2019,308679,308679,308679,0,308678,0,0,0,0,True,True,PASS_INDEX_CHECKS,Index metadata checks pass; semantic alignment...
5,2020,269853,269853,269853,0,269852,0,0,0,0,True,True,PASS_INDEX_CHECKS,Index metadata checks pass; semantic alignment...
6,2021,259625,259625,259625,0,259624,0,0,0,0,True,True,PASS_INDEX_CHECKS,Index metadata checks pass; semantic alignment...
7,2022,260490,260490,260490,0,260489,0,0,0,0,True,True,PASS_INDEX_CHECKS,Index metadata checks pass; semantic alignment...
8,2023,233307,233307,233307,0,233306,0,0,0,0,True,True,PASS_INDEX_CHECKS,Index metadata checks pass; semantic alignment...
9,2024,264946,264946,264946,0,264945,0,0,0,0,True,True,PASS_INDEX_CHECKS,Index metadata checks pass; semantic alignment...


Zarr metadata and sampled-row read checks


,store,zarr_exists,expected_rows,arrays,arrays_axis0_match,sample_reads_ok,sample_reads_tested,array_shapes,error
0,MODIS 2015,True,329903,3,True,9,9,"done=(329903,); evi=(329903, 27, 27); lst=(329...",None
1,Sentinel-1 2015,True,329903,8,True,24,24,"hh=(329903, 13, 13); hv=(329903, 13, 13); obs_...",None
2,MODIS 2016,True,394790,3,True,9,9,"done=(394790,); evi=(394790, 27, 27); lst=(394...",None
3,Sentinel-1 2016,True,394790,8,True,24,24,"hh=(394790, 13, 13); hv=(394790, 13, 13); obs_...",None
4,MODIS 2017,True,369953,3,True,9,9,"done=(369953,); evi=(369953, 27, 27); lst=(369...",None
5,Sentinel-1 2017,True,369953,8,True,24,24,"hh=(369953, 13, 13); hv=(369953, 13, 13); obs_...",None
6,MODIS 2018,True,344806,3,True,9,9,"done=(344806,); evi=(344806, 27, 27); lst=(344...",None
7,Sentinel-1 2018,True,344806,8,True,24,24,"hh=(344806, 13, 13); hv=(344806, 13, 13); obs_...",None
8,MODIS 2019,True,308679,3,True,9,9,"done=(308679,); evi=(308679, 27, 27); lst=(308...",None
9,Sentinel-1 2019,True,308679,8,True,24,24,"hh=(308679, 13, 13); hv=(308679, 13, 13); obs_...",None



Interpretation:
- Any FAIL means fix the corresponding mismatch before training with that raster.
- PASS_STRUCTURAL_ONLY for MODIS is NOT proof that row i in samples.parquet is the patch at Zarr row i.
- A valid Sentinel-1 row_index is only trustworthy if its producer records the same row index while writing the Zarr arrays.


In [2]:
# Search project source code for the MODIS/Sentinel raster writer logic.
# This is read-only; it skips data folders and virtual environments.

PROJECT_ROOT = ROOT.parents[1] if len(ROOT.parents) >= 2 else ROOT
SKIP_DIRS = {".git", ".venv", "venv", "env", "__pycache__", ".ipynb_checkpoints",
             "node_modules", "site-packages", "Data", "data", "outputs", "cache"}
SOURCE_SUFFIXES = {".py", ".ipynb"}
KEYWORDS = ("evi_lst_patch_dataset", "patches.zarr", "samples.parquet",
            "s1_nearest_valid_", "samples_nearest_valid_")

source_hits = []
scanned = 0
for dirpath, dirnames, filenames in os.walk(PROJECT_ROOT):
    dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS]
    for filename in filenames:
        p = Path(dirpath) / filename
        if p.suffix.lower() not in SOURCE_SUFFIXES:
            continue
        try:
            if p.stat().st_size > 15_000_000:
                continue
            if p.suffix.lower() == ".ipynb":
                obj = json.loads(p.read_text(encoding="utf-8"))
                chunks = []
                for i, cell in enumerate(obj.get("cells", [])):
                    if cell.get("cell_type") == "code":
                        src = "".join(cell.get("source", []))
                        if any(k.lower() in src.lower() for k in KEYWORDS):
                            chunks.append((f"cell {i}", src))
            else:
                chunks = [("file", p.read_text(encoding="utf-8", errors="replace"))]
            scanned += 1
            for where, src in chunks:
                low = src.lower()
                # Prefer code that mentions the raster output plus the sample sidecar.
                if ("patches.zarr" in low or "evi_lst_patch_dataset" in low or
                    "s1_nearest_valid_" in low or "samples_nearest_valid_" in low):
                    lines = src.splitlines()
                    indices = [i for i, line in enumerate(lines)
                               if any(k.lower() in line.lower() for k in KEYWORDS)]
                    for i in indices[:3]:
                        a, b = max(0, i-5), min(len(lines), i+9)
                        snippet = chr(10).join(f"{j+1}: {lines[j]}" for j in range(a, b))
                        source_hits.append({"file": str(p), "location": where,
                                            "trigger_line": i+1, "snippet": snippet})
        except Exception as exc:
            print(f"Skipped unreadable source {p}: {type(exc).__name__}: {exc}")

# Keep output manageable while preserving candidate file locations and code context.
print(f"Searched source files under: {PROJECT_ROOT}")
print(f"Source files scanned: {scanned}; matching code snippets: {len(source_hits)}")
if source_hits:
    seen = set()
    for hit in source_hits:
        key = (hit["file"], hit["location"], hit["trigger_line"])
        if key in seen:
            continue
        seen.add(key)
        print(); print("="*90)
        print(f"{hit['file']} ({hit['location']}, line {hit['trigger_line']})")
        print(hit["snippet"])
        if len(seen) >= 25:
            print(); print("Output capped at 25 snippets. Narrow KEYWORDS or rerun per file if needed.")
            break
else:
    print("No writer source was found automatically. In that case, MODIS physical row ordering cannot be proven from the sidecar and row counts alone.")

con_audit.close()


Searched source files under: /home/mayur/Code/UG_SM
Source files scanned: 53; matching code snippets: 36

/home/mayur/Code/UG_SM/Sentinel_worker_1.ipynb (cell 27, line 25)
20: 
21: 
22: def run_year(year):
23:     samples_year = prepare_year_metadata(year)
24: 
25:     metadata_path = FINAL_OUTPUT_DIR / f"samples_nearest_valid_{year}.parquet"
26:     samples_year.to_parquet(metadata_path, index=False)
27: 
28:     zarr_path = FINAL_OUTPUT_DIR / f"s1_nearest_valid_{year}.zarr"
29:     root = create_year_zarr(zarr_path, len(samples_year))
30: 
31:     done = np.asarray(root["sample_done"][:], dtype=bool)
32: 
33:     cluster_ranges = (

/home/mayur/Code/UG_SM/Sentinel_worker_1.ipynb (cell 27, line 28)
23:     samples_year = prepare_year_metadata(year)
24: 
25:     metadata_path = FINAL_OUTPUT_DIR / f"samples_nearest_valid_{year}.parquet"
26:     samples_year.to_parquet(metadata_path, index=False)
27: 
28:     zarr_path = FINAL_OUTPUT_DIR / f"s1_nearest_valid_{year}.zarr"
29:     root = c

## What counts as verified?

- **Structural pass:** every Zarr array has the expected sample dimension and can read representative first/middle/last sample slices.
- **Identity pass:** the sidecar's sample IDs, dates, station IDs, years, and row indices agree with the canonical sample table and store sizes.
- **Semantic raster alignment (the critical part):** the producer code shows each sample's metadata and its patch are written in the same loop/order, or an explicit `sample_id -> row_index` is recorded at write time. If the source code does not prove this for MODIS, rebuild the MODIS index while reprocessing patches or regenerate the Zarr and sidecar together. Do not infer it from equal counts.